# LegalIR Task 1: Kaggle 2×T4 CUDA Smoke Gate (B1.1)
## UIT Data Science Challenge 2026 — Reproducible Training Workflow
**Pinned Git Commit:** `5b00badd654caf04a855407b95c0b80d859b1986`

### Smoke Test Invariants:
- **Enforces Kaggle Dual-T4 GPU Topology** (`cuda:0` Dense + `cuda:1` Reranker).
- Consumes canonical Kaggle dataset: `/kaggle/input/legalir-task1-clean-data`.
- Mines real 50-query official evidence pairs (no synthetic dummy text).
- Validates `BAAI/bge-reranker-v2-m3` + LoRA forward/backward pass with `float16`.
- Asserts finite loss ($L < \infty$) and parameter update ($\Delta w > 0$).
- Emits `kaggle_t4x2_report.json` with verdict `PASS`.


In [ ]:
# ==============================================================================
# Cell 1: Hardware Preflight & GPU Verification
# ==============================================================================
import os
import sys
import subprocess
import torch

print(f"[+] Python Version : {sys.version.split()[0]}")
print(f"[+] PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "CUDA not detected. Enable GPU accelerator in Kaggle settings."
count = torch.cuda.device_count()
print(f"[+] Visible CUDA Devices: {count}")
for i in range(count):
    prop = torch.cuda.get_device_properties(i)
    vram = prop.total_memory / (1024**3)
    print(f"    - GPU {i}: {prop.name} | Total VRAM: {vram:.2f} GB")
assert count >= 2, f"Kaggle Dual-T4 Gate requires >= 2 CUDA devices (found {count})."

try:
    from kaggle_secrets import UserSecretsClient
    for sec_key in ['HF_TOKEN_WRITE', 'HF_TOKEN', 'HF_TOKEN_READ']:
        tok = UserSecretsClient().get_secret(sec_key)
        if tok and str(tok).startswith('hf_'):
            os.environ['HF_TOKEN'] = tok
            break
except Exception:
    pass

hf_candidates = [os.environ.get('HF_TOKEN_WRITE'), os.environ.get('HF_TOKEN'), os.environ.get('HF_TOKEN_READ')]
hf_tok = next((t for t in hf_candidates if t and str(t).startswith('hf_')), None)
if hf_tok:
    os.environ['HF_TOKEN'] = hf_tok
    try:
        from huggingface_hub import HfApi
        u_name = HfApi(token=hf_tok).whoami().get('name', 'unknown')
        print(f'[+] HF_TOKEN verified (authenticated as @{u_name} for model downloads & uploads).')
    except Exception:
        print('[+] HF_TOKEN active in environment for model downloads & uploads.')


In [ ]:
# ==============================================================================
# Cell 2: Repository Clone & Exact Git Commit Checkout
# ==============================================================================
from pathlib import Path

EXPECTED_COMMIT = os.environ.get("LEGALIR_COMMIT_SHA") or "5b00badd654caf04a855407b95c0b80d859b1986"
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR = WORK_DIR / "LegalIR"

if not REPO_DIR.exists():
    if (Path.cwd() / "src").is_dir() and (Path.cwd() / "scripts").is_dir():
        REPO_DIR = Path.cwd().resolve()
    else:
        print(f"[*] Cloning repository to {REPO_DIR}...")
        subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(REPO_DIR)], check=True)

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", EXPECTED_COMMIT], cwd=REPO_DIR, check=False)
    print(f"[*] Checking out exact commit: {EXPECTED_COMMIT} (detached HEAD)...")
    res = subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[*] Checkout fallback: unshallowing repository...")
        subprocess.run(["git", "fetch", "--unshallow", "origin"], cwd=REPO_DIR, check=False)
        subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"[+] Working in: {REPO_DIR}")


In [ ]:
# ==============================================================================
# Cell 3: Package Verification & Compatibility Setup
# Do NOT reinstall torch (Kaggle CUDA build). Install only missing wheels.
# ==============================================================================
import subprocess
import sys

needed_packages = []
for pkg, imp in [("peft", "peft"), ("pyvi", "pyvi"), ("pyarrow", "pyarrow"), ("bm25s", "bm25s"), ("huggingface_hub", "huggingface_hub")]:
    try:
        __import__(imp)
    except ImportError:
        needed_packages.append(pkg)

if needed_packages:
    print(f"[*] Installing missing packages: {needed_packages}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location"] + needed_packages, check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
sys.modules["torchao"] = None
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass
print("[+] Dependency preflight completed.")


In [ ]:
# ==============================================================================
# Cell 4: Execute Kaggle Dual-T4 Gate (scripts/run_kaggle_smoke.py -> scripts/gates/run_kaggle_t4x2.py)
# ==============================================================================
from src.data.canonical import discover_canonical_dataset_dir
from scripts.run_kaggle_smoke import run_kaggle_smoke

dataset_dir = discover_canonical_dataset_dir()
print(f"[+] Discovered Canonical Dataset: {dataset_dir}")

output_dir = WORK_DIR / "artifacts/task1/gates"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"[*] Running Kaggle Dual-T4 CUDA Gate on {dataset_dir}...")
report = run_kaggle_smoke(
    dataset_dir=dataset_dir,
    output_dir=output_dir,
    target_sha=EXPECTED_COMMIT,
    mock=False,
)
print(f"[+] Gate execution completed with verdict: {report.get('verdict')}")


In [ ]:
# ==============================================================================
# Cell 5: Assert Gate PASS
# ==============================================================================
import json

report_path = output_dir / "kaggle_t4x2_report.json"
assert report_path.is_file(), f"Report missing: {report_path}"
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report, indent=2))
assert report.get("verdict") == "PASS", f"Kaggle Dual-T4 Gate failed: {report}"
print("\n=================================================================")
print("[+] KAGGLE 2×T4 CUDA SMOKE GATE PASSED. READY FOR COLAB T4 GATE.")
print("=================================================================")
